In [ ]:
import OrcFxAPI
import numpy as np
from scipy.signal import find_peaks


# ============================================================
# MODEL PADEN
# ============================================================

model_path_fixed_original = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_fixed_120s_noaddedmass.dat"
model_path_spring_original = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_spring_120s_addedmass.dat"


# ============================================================
# PERIODEBEPALING
# Gebruik jouw eigen calculate_period als die al eerder staat.
# Anders wordt deze gebruikt.
# ============================================================

def calculate_period(time, signal):
    time = np.asarray(time)
    signal = np.asarray(signal)

    mask = np.isfinite(time) & np.isfinite(signal)
    time = time[mask]
    signal = signal[mask]

    if len(time) < 3:
        return None

    signal = signal - np.mean(signal)

    peaks, _ = find_peaks(signal)

    if len(peaks) < 2:
        return None

    peak_times = time[peaks]
    periods = np.diff(peak_times)

    return float(np.mean(periods))


# ============================================================
# HULPFUNCTIES
# ============================================================

def reset_added_mass_and_inertia(aeroaddedmass):
    aeroaddedmass.HydrodynamicMassX = 0.0
    aeroaddedmass.HydrodynamicMassY = 0.0
    aeroaddedmass.HydrodynamicMassZ = 0.0

    aeroaddedmass.HydrodynamicInertiaX = 0.0
    aeroaddedmass.HydrodynamicInertiaY = 0.0
    aeroaddedmass.HydrodynamicInertiaZ = 0.0


def set_zero_damping(floatertype):
    floatertype.OtherDampingLinearCoeffx = 0.0
    floatertype.OtherDampingLinearCoeffy = 0.0
    floatertype.OtherDampingLinearCoeffz = 0.0
    floatertype.OtherDampingLinearCoeffRx = 0.0
    floatertype.OtherDampingLinearCoeffRy = 0.0
    floatertype.OtherDampingLinearCoeffRz = 0.0

    floatertype.OtherDampingQuadraticCoeffx = 0.0
    floatertype.OtherDampingQuadraticCoeffy = 0.0
    floatertype.OtherDampingQuadraticCoeffz = 0.0
    floatertype.OtherDampingQuadraticCoeffRx = 0.0
    floatertype.OtherDampingQuadraticCoeffRy = 0.0
    floatertype.OtherDampingQuadraticCoeffRz = 0.0


def set_initial_condition(floaters, dof):
    # Alles eerst naar nul
    floaters.InitialX = 0.0
    floaters.InitialY = 0.0
    floaters.InitialZ = 0.0
    floaters.InitialHeel = 0.0
    floaters.InitialTrim = 0.0
    floaters.InitialHeading = 0.0

    # Decay per DOF
    if dof == "heave":
        floaters.InitialZ = 2.0          # m
    elif dof == "roll":
        floaters.InitialHeel = 2.0       # deg
    elif dof == "pitch":
        floaters.InitialTrim = 2.0       # deg
    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def get_signal(floaters, dof):
    if dof == "heave":
        return floaters.TimeHistory("Z")
    elif dof == "roll":
        return floaters.TimeHistory("Rotation 1")
    elif dof == "pitch":
        return floaters.TimeHistory("Rotation 2")
    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def run_period_without_added_mass(model_path, dof):
    model = OrcFxAPI.Model(model_path)

    floaters = model["floaters"]
    floatertype = model["Floatertype"]
    aeroaddedmass = model["aeroaddedmass"]

    # reset_added_mass_and_inertia(aeroaddedmass)
    set_zero_damping(floatertype)
    set_initial_condition(floaters, dof)

    # model.CalculateStatics()
    model.RunSimulation()

    time = model.general.TimeHistory("Time")
    signal = get_signal(floaters, dof)

    T = calculate_period(time, signal)

    return T


# ============================================================
# RUN ALLE BASISPERIODES ZONDER ADDED MASS/INERTIA
# ============================================================

models = {
    "fixed": model_path_fixed_original,
    "spring": model_path_spring_original,
}

dofs = ["heave", "roll", "pitch"]

baseline_periods = {}

for model_name, model_path in models.items():
    baseline_periods[model_name] = {}

    print("\n" + "=" * 60)
    print(f"MODEL: {model_name.upper()}")
    print("=" * 60)

    for dof in dofs:
        T = run_period_without_added_mass(model_path, dof)
        baseline_periods[model_name][dof] = T

        print(f"{dof:5s} periode zonder added mass/inertia = {T:.6f} s")


print("\n" + "=" * 60)
print("SAMENVATTING")
print("=" * 60)

for model_name in baseline_periods:
    print(f"\n{model_name.upper()}")
    for dof in baseline_periods[model_name]:
        print(f"  {dof:5s}: {baseline_periods[model_name][dof]:.6f} s")



MODEL: FIXED
heave periode zonder added mass/inertia = 3.800000 s
roll  periode zonder added mass/inertia = 6.565000 s
pitch periode zonder added mass/inertia = 6.565000 s

MODEL: SPRING
heave periode zonder added mass/inertia = 3.919355 s
roll  periode zonder added mass/inertia = 7.017647 s
pitch periode zonder added mass/inertia = 7.017647 s

SAMENVATTING

FIXED
  heave: 3.800000 s
  roll : 6.565000 s
  pitch: 6.565000 s

SPRING
  heave: 3.919355 s
  roll : 7.017647 s
  pitch: 7.017647 s
